# Lab 4: HR Candidate Profile Parser
Parse a resume into a clean JSON object with LangChain's `StructuredOutputParser`.

**Fixes compared with the first version**
- `generate_text` now returns **only the model's answer** (the old version returned prompt + answer, so `matches[-1]` could pick the *example* JSON from the prompt).
- Uses `max_new_tokens` (answer length) instead of `max_length` (prompt + answer), so long CVs are not cut off.
- Uses the model's chat template, and a more tolerant JSON extraction + normalisation step.
- Runs on the lab's sample text by default, so the notebook contains no personal data.

In [ ]:
!pip install -q transformers==4.52.4 langchain-classic langchain-core pypdf

In [ ]:
import json
import re

import torch
from langchain_classic.output_parsers import ResponseSchema, StructuredOutputParser
from langchain_core.prompts import PromptTemplate
from pypdf import PdfReader
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
model_name = "mistralai/Mistral-Nemo-Instruct-2407"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

In [ ]:
def generate_text(prompt, max_new_tokens=900):
    """Return only the newly generated text (no prompt)."""
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    output = model.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

## Schema and prompt

In [ ]:
response_schemas = [
    ResponseSchema(name="full_name", description="Candidate's full name"),
    ResponseSchema(name="email", description="Candidate's email address"),
    ResponseSchema(name="education",
                   description="List of objects with keys: degree, institution, year (year as an integer)",
                   type="List[dict]"),
    ResponseSchema(name="skills", description="List of skill strings", type="List[string]"),
    ResponseSchema(name="experience",
                   description="List of objects with keys: role, company, years (years as a string like '2020-2023')",
                   type="List[dict]"),
]
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()

In [ ]:
cv_template = """You are an HR assistant that parses resumes into structured JSON.

Respond ONLY with one JSON object, formatted as follows:
{format_instructions}

Rules:
- Use only information that appears in the resume. Never invent values.
- If a field is missing, use "" for text or [] for lists.
- "year" is a 4-digit integer.
- "years" uses year numbers only, like "2020-2023" or "2023-Present". Never write month names.
- Put one skill per list item.

<example>
Input:
Jane Doe - jane@mail.com
Education: M.Sc. Physics, Stanford, 2019
Skills: Java, SQL
Experience:
- Engineer at Tesla (2019-2022)

Output:
```json
{{
  "full_name": "Jane Doe",
  "email": "jane@mail.com",
  "education": [{{"degree": "M.Sc. Physics", "institution": "Stanford", "year": 2019}}],
  "skills": ["Java", "SQL"],
  "experience": [{{"role": "Engineer", "company": "Tesla", "years": "2019-2022"}}]
}}
```
</example>

Now extract from this resume:
<resume>
{cv_text}
</resume>
"""
prompt_template = PromptTemplate(template=cv_template, input_variables=["cv_text", "format_instructions"])

## Parsing helpers

In [ ]:
def extract_json_text(response):
    fenced = re.findall(r"```(?:json)?\s*(\{.*?\})\s*```", response, re.DOTALL)
    if fenced:
        return fenced[-1]
    start, end = response.find("{"), response.rfind("}")
    if start == -1 or end <= start:
        raise ValueError("The model did not return JSON.")
    return response[start:end + 1]


def clean_years(value):
    # 'Jan 2020 - Mar 2023' -> '2020–2023' (year numbers only, en dash as in the lab's expected output)
    tokens = re.findall(r"\d{4}|present|current|now", str(value or ""), re.I)
    return "–".join(t if t.isdigit() else "Present" for t in tokens[:2]) or str(value or "").strip()


def clean_year(value):
    match = re.search(r"\d{4}", str(value or ""))
    return int(match.group()) if match else None


def parse_cv(response):
    data = json.loads(extract_json_text(response))
    for key in ("education", "skills", "experience"):
        data.setdefault(key, [])
    data.setdefault("full_name", "")
    data.setdefault("email", "")
    result = output_parser.parse("```json\n" + json.dumps(data) + "\n```")  # schema check
    for row in result["education"]:
        row["year"] = clean_year(row.get("year"))
    for row in result["experience"]:
        row["years"] = clean_years(row.get("years"))
    return result


def show(result):
    print(json.dumps(result, indent=2, ensure_ascii=False))

## Run on the lab's sample

In [ ]:
cv_text = """John Smith – john.smith@email.com
Education: B.Sc. Computer Science, MIT, 2020
Skills: Python, Machine Learning, Data Analysis
Experience:
- Software Engineer at Google (2020–2023)
- Data Scientist at OpenAI (2023–Present)"""

prompt = prompt_template.format(cv_text=cv_text, format_instructions=format_instructions)
response = generate_text(prompt)
show(parse_cv(response))

## Optional: run on a PDF resume
Keep real CVs out of the public repository (they contain personal data).

In [ ]:
# pdf_path = "/kaggle/input/your-dataset/resume.pdf"
# cv_text = "\n".join(page.extract_text() or "" for page in PdfReader(pdf_path).pages)
# prompt = prompt_template.format(cv_text=cv_text, format_instructions=format_instructions)
# show(parse_cv(generate_text(prompt)))